In [20]:
import pandas as pd
from pathlib import Path

PAPER_PATH = Path("../papers/")

## Investigate Initial Data Pull

We're validating July's parsed papers to confirm the parser worked as expected and that entries are genuine academic papers submitted to arXiv, excluding other content types such as full-length books or dissertations.

In [19]:
df = pd.read_parquet(PAPER_PATH / "parsed_papers.parquet")
df.head(10)

,paper_path,paper_id,page_number,text
0,papers\2607.00283v1.pdf,2607.00283v1,0,# **What’s Hidden Matters: Identifying Plannin...
1,papers\2607.00283v1.pdf,2607.00283v1,1,\n\n<!-- Start of picture text -->\nnuScenes +...
2,papers\2607.00283v1.pdf,2607.00283v1,2,"driving stack as a causal graph, as in DriveLM..."
3,papers\2607.00283v1.pdf,2607.00283v1,3,_3) Structured Annotation Generation:_ The key...
4,papers\2607.00283v1.pdf,2607.00283v1,4,as described in Section III-C.3. Our final dat...
5,papers\2607.00283v1.pdf,2607.00283v1,5,TABLE II: Supervised fine-tuned model performa...
6,papers\2607.00283v1.pdf,2607.00283v1,6,InternVL3.5-8B shows mixed results: accuracy a...
7,papers\2607.00283v1.pdf,2607.00283v1,7,all model families and scales. Our results sho...
8,papers\2607.00283v1.pdf,2607.00283v1,8,"- [36] Zhenhua Xu, Yujia Zhang, Enze Xie, Zhen..."
9,papers\2607.00292v1.pdf,2607.00292v1,0,# An LLM-Based Framework for Intent-Driven Net...


## Validation of Total Papers Processed
We downloaded 4255 papers and can see that there are 4255 that were successfully parsed, suggesting that each paper at least had one page successfully parsed.

In [7]:
df['paper_id'].nunique()

4255

## Validation of Text Missingness

Let's validate the parsed text for completeness. If the paper was successfully parsed it should:
- Not have an NA/NULL value in the `text` column
- Not have an empty string in the `text` column
- Not have strings only non-alphabetical characters in `text` column

> **NOTE:** Of course, for each of these assumptions there will be exceptions. A page with no text should, in fact, be NULL or empty string; a page with only images or figures may not containt alphabetical characters, etc. **For each paper that violates our assumptions, we will manually validate to classify as true exception or parsing error.**

In [51]:
pages_na = (df['text'].isna())
pages_no_text = ( df['text'] == '')
pages_only_space = (df['text'].str.isspace())
pages_no_alpha = (~df['text'].str.contains(r'[a-zA-Z]', na=False))


print(f"Missing pages: {pages_na.sum()}")
print(f"Pages with empty text: {pages_no_text.sum()}")
print(f"Pages only spaces: {pages_only_space.sum()}")
print(f"Pages with no letters: {pages_no_alpha.sum()}")

Missing pages: 0
Pages with empty text: 16
Pages only spaces: 1
Pages with no letters: 21


In [52]:
paper_ids_no_text = df[pages_no_text]['paper_id'].tolist()
paper_ids_only_space = df[pages_only_space]['paper_id'].tolist()
paper_ids_no_alpha = df[pages_no_alpha]['paper_id'].tolist()

### Papers with Empty Strings

In [59]:
empty_string_df = df[pages_no_text]
empty_string_df['flagged_pages'] = pages_no_text
empty_string_df = df.merge(empty_string_df[['paper_id', 'page_number', 'flagged_pages']],
         how='left',
         on=['paper_id', 'page_number'])


empty_string_df[empty_string_df['paper_id'].isin(paper_ids_no_text)].head(10)

,paper_path,paper_id,page_number,text,flagged_pages
34752,papers\2607.11938v1.pdf,2607.11938v1,0,\n\n<!-- Start of picture text -->\nSCIENC<br>...,<NA>
34753,papers\2607.11938v1.pdf,2607.11938v1,1,### **Mathematics of Data Science** \n\n```\nP...,<NA>
34754,papers\2607.11938v1.pdf,2607.11938v1,2,# **Contents** \n\n|**Co**|**nten**|**ts**|**i...,<NA>
34755,papers\2607.11938v1.pdf,2607.11938v1,3,|ii|5.1|Contents<br>PageRank . . . . . . . . ....,<NA>
34756,papers\2607.11938v1.pdf,2607.11938v1,4,|Content|s|iii|\n|---|---|---|\n|**11 Lar**|**...,<NA>
34757,papers\2607.11938v1.pdf,2607.11938v1,5,,True
34758,papers\2607.11938v1.pdf,2607.11938v1,6,## **Chapter 0** \n\n# **Notes on this Version...,<NA>
34759,papers\2607.11938v1.pdf,2607.11938v1,7,,True
34760,papers\2607.11938v1.pdf,2607.11938v1,8,## **Chapter 1** \n\n# **Introduction** \n\n##...,<NA>
34761,papers\2607.11938v1.pdf,2607.11938v1,9,1. Introduction \n\n4 \n\nis revolutionizing r...,<NA>


### Papers with Only Spaces

Not really a concern since we only saw one.

In [74]:
only_spaces_df = df[pages_only_space]
only_spaces_df['flagged_pages'] = pages_only_space
only_spaces_df = df.merge(only_spaces_df[['paper_id', 'page_number', 'flagged_pages']],
         how='left',
         on=['paper_id', 'page_number'])


only_spaces_df[only_spaces_df['paper_id'].isin(paper_ids_only_space)].head(30)

,paper_path,paper_id,page_number,text,flagged_pages
48837,papers\2607.16903v1.pdf,2607.16903v1,0,# **A Method for Learning Value Systems in Gen...,<NA>
48838,papers\2607.16903v1.pdf,2607.16903v1,1,"decision-making contexts, where utility or rew...",<NA>
48839,papers\2607.16903v1.pdf,2607.16903v1,2,may expend computational resources on approach...,<NA>
48840,papers\2607.16903v1.pdf,2607.16903v1,3,"Importantly, our reward functions do not aim t...",<NA>
48841,papers\2607.16903v1.pdf,2607.16903v1,4,in terms of the final representativeness of va...,<NA>
48842,papers\2607.16903v1.pdf,2607.16903v1,5,parametrized with _ω ∈_ R<sup>_m_</sup> throug...,<NA>
48843,papers\2607.16903v1.pdf,2607.16903v1,6,"To satisfy the previous constraints, we use au...",<NA>
48844,papers\2607.16903v1.pdf,2607.16903v1,7,**Training details.** The datasets are split i...,<NA>
48845,papers\2607.16903v1.pdf,2607.16903v1,8,|Method|Helpfulness|Honesty|Truthfulness|Instr...,<NA>
48846,papers\2607.16903v1.pdf,2607.16903v1,9,|Method|Prompt Following|Objectivity|Clarity|I...,<NA>


### Papers with No Word Characters


In [75]:
no_alphabet_df = df[pages_no_alpha]
no_alphabet_df['flagged_pages'] = pages_no_alpha
no_alphabet_df = df.merge(no_alphabet_df[['paper_id', 'page_number', 'flagged_pages']],
         how='left',
         on=['paper_id', 'page_number'])


no_alphabet_df[no_alphabet_df['paper_id'].isin(paper_ids_no_alpha)].head(25)

,paper_path,paper_id,page_number,text,flagged_pages
3527,papers\2607.01511v1.pdf,2607.01511v1,0,\n\n<!-- Start of picture text -->\n© Manifold...,<NA>
3528,papers\2607.01511v1.pdf,2607.01511v1,1,"At the same time, unlabeled questions are ofte...",<NA>
3529,papers\2607.01511v1.pdf,2607.01511v1,2,"100% on MultiArith. In terms of accuracy, Semi...",<NA>
3530,papers\2607.01511v1.pdf,2607.01511v1,3,Reasoning quality is not fully captured by fin...,<NA>
3531,papers\2607.01511v1.pdf,2607.01511v1,4,This shift creates a new problem. A generated ...,<NA>
3532,papers\2607.01511v1.pdf,2607.01511v1,5,A general Semi-CoT method contains three steps...,<NA>
3533,papers\2607.01511v1.pdf,2607.01511v1,6,### **3.3 Pseudo-CoT Generation** \n\nFor each...,<NA>
3534,papers\2607.01511v1.pdf,2607.01511v1,7,"In the current prompt-level implementation, _w...",<NA>
3535,papers\2607.01511v1.pdf,2607.01511v1,8,**Algorithm 1** Semi-CoT: Semi-supervised Chai...,<NA>
3536,papers\2607.01511v1.pdf,2607.01511v1,9,When all sampled CoTs lead to the same answer ...,<NA>


## Conclusions From Text Missingness Analysis
- Text-loss issues affect only 38 of ~89,000 parsed pages (~0.04%) — the parser is reliable.
- Nearly all flagged pages are expected structural artifacts: trailing blank pages, page-number-only pages, or table-of-contents formatting.
- The one genuine anomaly, 2607.11938v1, isn't a parsing erorr -- it's a textbook that shouldn't have been included as a "paper" in the first place. This foreshadows the page-count filtering below


### Distribution of Papers by Page Count

Most (90%) of papers have 36 pages or less which seems resonable. However, there is a small chunk of papers with 100+ pages which is suspicious. Let's further investigate these papers to make sure that we are only downloading papers.

In [37]:
df.groupby(['paper_id'])['page_number'].count().describe(percentiles=[.25, .5, .75,.8,.9,.95,.99])

count    4255.000000
mean       20.942656
std        19.620938
min         2.000000
25%        11.000000
50%        17.000000
75%        26.000000
80%        28.000000
90%        36.000000
95%        45.000000
99%        76.460000
max       547.000000
Name: page_number, dtype: float64

## Papers By Page Count

After manual review we have found a couple of different types of documents among the "papers" we downloaded:
- Books `(>400 pgs)`
- Dissertations `(200 - 400 pgs)`
- Academic papers `(<200 pgs)`

Since the focus is on novel research we will be filtering out books (or documents with 400+ pgs) whose focus is more on reviewing and compiling exisitng research.

## Summary
The parser performed reliably: all 4255 downloaded papers parsed successfully, and text-level issues affected under 0.05% of pages, nearly all of which were expected artifacts (blank/page-number-only pages) rather than genuine failures.

The real filtering concern isn't parsing quality — it's that a small number of downloaded documents aren't novel research at all (textbooks, dissertations). Page count is a strong proxy for this, and the two anomalies caught in the missingness analysis both correspond to non-paper documents identified independently via page count.

Next steps: define and apply a page-count cutoff to exclude books, manually spot-check the 200–400 page dissertation band before deciding whether to include/exclude, and re-validate paper count after filtering.